# ⚡ Crushing Stiffness: The Deterministic PINNsFormer
Welcome to the deterministic foundation of the **PINNsFormer**. Before we can measure the uncertainty of a model, we must ensure the core physics engine is flawless. 

In this notebook, we are abandoning standard Multi-Layer Perceptrons (MLPs) and numerical ODE solvers. Instead, we are utilizing **Sequence-to-Sequence (Seq2Seq)** modeling and **Wavelet Activations** to conquer the stiffness of the Hodgkin-Huxley (HH) equations.

---

## 1. The Data Paradigm Shift: Time as a Sequence
Standard Neural ODEs process time sequentially: $u_{t+1} = u_t + \text{Solver}(f(u))$. If the system is stiff (like an action potential), numerical errors compound at every step.

The PINNsFormer completely changes how we view time. We stop stepping forward and instead process the entire temporal trajectory as a single, holistic object.

To do this, we must reshape our data to fit the standard Transformer tensor format:
$$ \text{Shape: } (\text{Features}, \text{Sequence Length}, \text{Batch Size}) $$

By treating the entire 100ms simulation as a single sequence batch, the network can look at the past, present, and future simultaneously.

## 2. The Architecture: Attention and Wavelets
Standard Physics-Informed Neural Networks (PINNs) suffer from **Spectral Bias**—they naturally prefer learning low-frequency, smooth functions. The HH equations, however, are dominated by high-frequency, violent voltage spikes. If a standard MLP tries to learn this, it will either smooth out the spike or oscillate uncontrollably. 

We solve this using a two-pronged architectural approach:

### A. Global Temporal Awareness (Self-Attention)
The Transformer's Self-Attention mechanism computes how every single time point relates to every other time point in the sequence:
$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$
This allows the network to instantly correlate the slow buildup of the gating variables at $t=10$ms with the explosive voltage spike at $t=35$ms, entirely bypassing the need for an ODE solver.

### B. The Anti-Stiffness Weapon (Mexican Hat Wavelet)
To defeat Spectral Bias, we replace standard global activations (like $\tanh$) inside the Feed-Forward block with the **Mexican Hat Wavelet**:
$$ \psi(x) = (1 - x^2) \exp\left(-\frac{x^2}{2}\right) $$
Unlike $\tanh$, which affects the output everywhere, a wavelet is localized. It stays near zero during the flat resting potential, and sharply "fires" only when the HH dynamics demand an action potential. The network learns to stretch and shift these wavelets to perfectly capture high-frequency stiff dynamics without corrupting the smooth areas.

## 3. The Seq2Seq Physics Loss (Finite Differences)
Because we have eliminated the ODE solver, we can no longer evaluate the physics at a single point in time and step forward. We must enforce the Hodgkin-Huxley physics directly across the entire predicted sequence at once.

### The Physics Residual
We need to know the predicted temporal derivative ($du/dt$) of our sequence. Since doing continuous auto-differentiation through complex Attention layers is computationally brutal, we use **Central Finite Differences** across our discrete sequence $U_{pred}$:
$$ \frac{d u_i}{dt} \approx \frac{u_{i+1} - u_{i-1}}{2\Delta t} $$

We pass our interior sequence points through the exact HH physical equations—let's call that mathematical operator $\mathcal{F}(u)$—and penalize any difference between our sequence's actual derivative and the theoretical derivative:
$$ \mathcal{L}_{phys} = \frac{1}{N} \sum_{i=2}^{N-1} \left\| \frac{u_{i+1} - u_{i-1}}{2\Delta t} - \mathcal{F}(u_i) \right\|^2 $$

*Note: Just like in the PI-NODE-SR framework, we still divide this residual by our Scale Factors ($s_j$) to ensure the massive voltage gradients do not drown out the tiny gating gradients!*

In [1]:
using  Flux,SciMLSensitivity, Optimization, OptimizationOptimisers, Statistics, Random, ComponentArrays, Zygote

In [2]:

using CSV, DataFrames

file_path = raw"E:\Neural_Spiking_Dynamics\notebooks\1_data_generation\single_spike_noisy_data.csv"


HH_data = CSV.read(file_path, DataFrame)



# 1. Correct the DataFrame mapping
df_ordered = DataFrame(
    timestamp = HH_data.timestamp,
    V = HH_data.V,
    n = HH_data.n, 
    m = HH_data.m,
    h = HH_data.h  
)

# 2. Extract into Float32 arrays for Lux.jl / Zygote.jl
t_train = Float32.(df_ordered.timestamp)

# 3. Extract states and transpose to get the required shape: (4, Total_Time_Steps)
z_synthetic = Float32.(Matrix(df_ordered[:, [:V, :n, :m, :h]]))'

println("Time array shape: ", size(t_train))
println("State matrix shape: ", size(z_synthetic))

Time array shape: (1469,)
State matrix shape: (4, 1469)


In [3]:
using Random
Random.seed!(42)
# Global sequence window configuration hyperparameters
const window_len = 50          # Fixed relative width for tracking windows
const stride = 5              # Window overlap step stride
const dt_step = 0.01f0         # Physical timestep interval

0.01f0

In [16]:
num_features = size(z_synthetic, 1) # 4 (V, n, m, h)
total_timesteps = size(z_synthetic, 2)          # This will be 4 (V, n, m, h)
# Calculate total possible windows
num_windows = div(total_timesteps - window_len, stride) + 1

284

In [17]:

z_windows = zeros(Float32, num_features, window_len, num_windows)

4×50×284 Array{Float32, 3}:
[:, :, 1] =
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0

[:, :, 2] =
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0

[:, :, 3] =
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.

In [18]:
# Save the absolute starting indices 'k' for dynamic loss tracking anchoring
window_start_indices = zeros(Int, num_windows)

for i in 1:num_windows
    start_idx = (i - 1) * stride + 1
    end_idx = start_idx + window_len - 1
    
    z_windows[:, :, i] = z_synthetic[:, start_idx:end_idx]
    window_start_indices[i] = start_idx
end



In [19]:
# Generate the standardized Relative Time coordinate matrix
# Crucial: Every single batch element sees identical relative coordinates starting from 0.0f0!
t_relative_base = Float32.(0:window_len-1) .* dt_step
t_relative_seq = reshape(t_relative_base, (1, window_len, 1)) # Projected broadcast form: (1, Seq_Len, 1)

println("--- WINDOWED DATA CONFIGURATION ---")
println("Total Rolling Windows Generated: ", num_windows)
println("Relative Time Input Shape       : ", size(t_relative_seq), " -> Resetting bound boundaries safely.")
println("State Tensor Tracking Shape     : ", size(z_windows), " -> (Features, Seq_Len, Batches)")

--- WINDOWED DATA CONFIGURATION ---
Total Rolling Windows Generated: 284
Relative Time Input Shape       : (1, 50, 1) -> Resetting bound boundaries safely.
State Tensor Tracking Shape     : (4, 50, 284) -> (Features, Seq_Len, Batches)


# -------------------------------------

In [20]:


# ==========================================
# CUSTOM ACTIVATION FUNCTION DEFINITION
# ==========================================

"""
    mexican_hat(x)

Custom non-monotonic Wavelet activation function designed to eliminate 
spectral bias and capture explosive high-frequency neuronal spikes.
"""
function mexican_hat(x)
    return (1.0f0 .- x.^2) .* exp.(-x.^2 ./ 2.0f0)
end


mexican_hat

In [21]:
struct PINNsFormerBlock
    layer_norm::LayerNorm
    attn::Flux.MultiHeadAttention
end

Flux.@layer PINNsFormerBlock

function PINNsFormerBlock(d_model::Int=32, nheads::Int=2)
    @assert d_model % nheads == 0 "d_model must be perfectly divisible by nheads."
    return PINNsFormerBlock(
        LayerNorm(d_model),
        Flux.MultiHeadAttention(d_model; nheads=nheads)
    )
end

function (m::PINNsFormerBlock)(x::AbstractArray{T, 3}) where T
    # x layout format: (d_model, Sequence_Length, Batch_Size)
    seq_len = size(x, 2)
    
    # 1. Generate Causal Attention Mask: Lower triangular matrix 
    # True indicates an allowed position; False indicates a masked value (-Inf)
    causal_mask = tril(ones(Bool, seq_len, seq_len))
    
    # 2. Process through attention utilizing the structural causal mask pass
    attn_result = m.attn(x, x, x; mask=causal_mask)
    
    # Extract element safely if a structural tuple is returned
    attn_out = attn_result isa Tuple ? first(attn_result) : attn_result
    
    # Add residual shortcut connection and apply normalization
    return m.layer_norm(x .+ attn_out)
end

In [22]:
struct HH_PINNsFormer
    embedder::Dense
    transformer_block::PINNsFormerBlock
    wavelet_ffn1::Dense
    wavelet_ffn2::Dense
    decoupler::Dense
end

Flux.@layer HH_PINNsFormer

function HH_PINNsFormer(; d_model::Int=32, nheads::Int=2, out_dim::Int=4)
    return HH_PINNsFormer(
        Dense(1 => d_model),                       # Spatiotemporal Relative Mixer
        PINNsFormerBlock(d_model, nheads),         # Causal Sequence Context Router
        Dense(d_model => d_model * 2),             # Wavelet expansion layer
        Dense(d_model * 2 => d_model),             # Wavelet compression layer
        Dense(d_model => out_dim)                  # Physical Output Decoupler Matrix
    )
end

function (nn::HH_PINNsFormer)(t_seq::AbstractArray{T, 3}) where T
    # Accept standardized relative time tensor: (1, Seq_Len, 1) or (1, Seq_Len, Batches)
    h0 = nn.embedder(t_seq)                        # Shape: (d_model, Seq_Len, Batches)
    h1 = nn.transformer_block(h0)                  # Shape: (d_model, Seq_Len, Batches)
    
    h2 = nn.wavelet_ffn1(h1)
    h2_act = mexican_hat(h2)
    h3 = nn.wavelet_ffn2(h2_act)                   # Shape: (d_model, Seq_Len, Batches)
    
    h_integrated = h1 .+ h3
    u_hat = nn.decoupler(h_integrated)            # Shape: (4, Seq_Len, Batches)
    
    return u_hat
end

In [24]:
# ==========================================
# SANITY CHECK & DIMENSIONAL VERIFICATION
# ==========================================
println("Initializing model instance with robust Tuple handling...")
println("Initializing dynamic causal model instance...")
model = HH_PINNsFormer(d_model=32, nheads=2, out_dim=4)

# Test forward pass with batched rolling window inputs
test_u_hat = model(t_relative_seq) # Leverages broadcasting across batch dimensions naturally!

println("\n--- CAUSAL MANIFOLD PASSPORT VERIFICATION ---")
println("Input Relative Time Shape : ", size(t_relative_seq))
println("Output Sequence Shape      : ", size(test_u_hat), " -> Mapped safely across batches.")


Initializing model instance with robust Tuple handling...
Initializing dynamic causal model instance...


LoadError: UndefVarError: `tril` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name also exists in LinearAlgebra.

In [ ]:
"""
    compute_physics_residuals(u_hat, dt)

Zygote-safe finite difference engine calculating Hodgkin-Huxley interior residuals.
Incorporates singularity prevention squeezes to prevent division-by-zero errors.
"""
function compute_physics_residuals(u_hat::AbstractArray{Float32, 3}, dt::Float32)
    # Extract states across time slices using functional array views
    V = @view u_hat[1, :, 1]
    n = @view u_hat[2, :, 1]
    m = @view u_hat[3, :, 1]
    h = @view u_hat[4, :, 1]

    # Central difference operations for internal tracking
    dV_dt = (V[3:end] .- V[1:end-2]) ./ (2.0f0 * dt)
    dn_dt = (n[3:end] .- n[1:end-2]) ./ (2.0f0 * dt)
    dm_dt = (m[3:end] .- m[1:end-2]) ./ (2.0f0 * dt)
    dh_dt = (h[3:end] .- h[1:end-2]) ./ (2.0f0 * dt)

    # Interior state matching arrays
    V_mid = @view V[2:end-1]
    n_mid = @view n[2:end-1]
    m_mid = @view m[2:end-1]
    h_mid = @view h[2:end-1]

    # Hodgkin-Huxley Constant Parameters
    C_m   = 1.0f0
    g_Na  = 120.0f0
    g_K   = 36.0f0
    g_L   = 0.3f0
    V_Na  = 115.0f0
    V_K   = -12.0f0
    V_L   = 10.6f0
    I_ext = 10.0f0

    # Voltage Equation Residual
    I_Na = g_Na .* (m_mid.^3) .* h_mid .* (V_mid .- V_Na)
    I_K  = g_K  .* (n_mid.^4) .* (V_mid .- V_K)
    I_L  = g_L  .* (V_mid .- V_L)
    res_V = dV_dt .- ((I_ext .- I_Na .- I_K .- I_L) ./ C_m)

    # Gating Variable Voltage-Dependent Functions (with Singularity Preventions)
    ϵ = 1.0f-5
    
    # Secure α_n denominator boundary
    num_n = 0.1f0 .- 0.01f0 .* V_mid
    den_n = exp.(1.0f0 .- 0.1f0 .* V_mid) .- 1.0f0
    α_n = num_n ./ (ifelse.(abs.(den_n) .< ϵ, ϵ, den_n))
    β_n = 0.125f0 .* exp.(-V_mid ./ 80.0f0)
    
    # Secure α_m denominator boundary
    num_m = 2.5f0 .- 0.1f0 .* V_mid
    den_m = exp.(2.5f0 .- 0.1f0 .* V_mid) .- 1.0f0
    α_m = num_m ./ (ifelse.(abs.(den_m) .< ϵ, ϵ, den_m))
    β_m = 4.0f0 .* exp.(-V_mid ./ 18.0f0)
    
    α_h = 0.07f0 .* exp.(-V_mid ./ 20.0f0)
    β_h = 1.0f0 ./ (exp.(3.0f0 .- 0.1f0 .* V_mid) .+ 1.0f0)

    # Gating Residual Tracks
    res_n = dn_dt .- (α_n .* (1.0f0 .- n_mid) .- β_n .* n_mid)
    res_m = dm_dt .- (α_m .* (1.0f0 .- m_mid) .- β_m .* m_mid)
    res_h = dh_dt .- (α_h .* (1.0f0 .- h_mid) .- β_h .* h_mid)

    return res_V, res_n, res_m, res_h
end

"""
    compute_total_loss(current_model, t_seq, z_data, dt; λ_phys, λ_ic)

Accepts an instantiated model mapping containing the active optimization parameters.
"""
function compute_total_loss(current_model, t_seq::AbstractArray{Float32, 3}, z_data::AbstractArray{Float32, 3}, dt::Float32; λ_phys::Float32=1.0f0, λ_ic::Float32=10.0f0)
    u_hat = current_model(t_seq)
    
    loss_data = mean((u_hat .- z_data).^2)
    
    res_V, res_n, res_m, res_h = compute_physics_residuals(u_hat, dt)
    loss_phys = mean(res_V.^2) + mean(res_n.^2) + mean(res_m.^2) + mean(res_h.^2)
    
    u_init_pred = u_hat[:, 1, 1]  
    u_init_true = z_data[:, 1, 1] 
    loss_ic = mean((u_init_pred .- u_init_true).^2)
    
    total_loss = loss_data + (λ_phys * loss_phys) + (λ_ic * loss_ic)
    
    return total_loss, loss_data, loss_phys, loss_ic
end

In [ ]:
# ==========================================
# LOCAL EXECUTION & STENCIL VERIFICATION
# ==========================================
dt_step = 0.01f0
println("Evaluating central difference stencil with dt = ", dt_step)

# Calling the newly compiled function name
f_V, f_n, f_m, f_h = compute_physics_residuals(test_u_hat, dt_step)

println("\n--- PHYSICS RESIDUAL PASSPORT ---")
println("Derivative Stencil Shape (f_V): ", size(f_V))
println("Derivative Stencil Shape (f_n): ", size(f_n))

if size(f_V) == (1467,)
    println("✅ Success: Residual matrices pass physical dimensional tests safely.")
else
    println("❌ Mismatch: Residual slicing failed boundary checks.")
end


In [ ]:
"""
    stateless_loss_wrapper(θ, p)

Pure stateless objective wrapper mapping optimization parameters explicitly 
via the reconstruction mapping function 're'.
"""
function stateless_loss_wrapper(θ::ComponentVector, p)
    # unpack parameters: (t_seq, z_data, dt, λ_phys, λ_ic)
    t_seq, z_data, dt, λ_phys, λ_ic = p
    
    # RECONSTRUCT structural model container with active parameters 
    local_model = re(θ)
    
    # Evaluate forward pass cleanly using explicit parameters
    loss_val, l_data, l_phys, l_ic = compute_total_loss(local_model, t_seq, z_data, dt; λ_phys=λ_phys, λ_ic=λ_ic)
    
    return loss_val, l_data, l_phys, l_ic
end

In [ ]:
println("Mapping architecture parameters to unified SciML optimization vectors...\n")

# Flatten model parameters and isolate the structural architecture mapping function 're'
flat_θ, re = Flux.destructure(model)
println("Total trainable parameters mapped into SciML vector: ", length(flat_θ))

# Configure fixed training parameters hyperparameter tuple
λ_phys_init = 1.0f0
λ_ic_init   = 10.0f0
opt_params  = (test_t_seq, z_seq, 0.01f0, λ_phys_init, λ_ic_init)


In [ ]:
# Construct the core optimization function profile, explicitly mounting AutoZygote
opt_func = Optimization.OptimizationFunction(stateless_loss_wrapper, Optimization.AutoZygote())

# Initialize the concrete Optimization Problem structure
opt_prob = Optimization.OptimizationProblem(opt_func, flat_θ, opt_params)

println("✅ Success: Optimization problem constructed with complete Zygote tracking structures.")


In [ ]:
const iter_count = Ref(0)

function optimization_callback(θ, loss, l_data, l_phys, l_ic)
    iter_count[] += 1
    if iter_count[] % 10 == 0 || iter_count[] == 1
        @info "Iteration Pipeline Update" Epoch=iter_count[] Total_Loss=loss Data_Loss=l_data Phys_Residual=l_phys IC_Anchor=l_ic
    end
    return false # Return true to trigger early termination if convergence is satisfied
